In [4]:
# Correct the LightRAG installation by installing 'lightrag-hku' at a specific version

!pip install lightrag-hku==1.4.9.11

First, we need to mount your Google Drive to access the `knowledge_graph` folder you've uploaded. Then, we will define the working directory and initialize LightRAG.

In [6]:
# 1. Montar tu Drive donde subiste la carpeta 'knowledge_graph2'
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [7]:
#The `WORKING_DIR` definition has been moved to the LightRAG initialization cell to ensure it's always available when needed.

In [11]:
import sys
!{sys.executable} -m pip install cohere

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 369.8/369.8 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 44.7 MB/s eta 0:00:00


Now, we will initialize LightRAG. Since your JSONs already contain the processed data, LightRAG will automatically detect them. **Do not use `rag.insert()` or reload PDFs.**

In [12]:
import os
import cohere
import asyncio
from dataclasses import dataclass
from lightrag import LightRAG

# 5. Creamos contenedores dinámicos para engañar a la línea 549 de la librería
@dataclass
class ContenedorLLM:
    func: callable
    max_token_size: int = 32768

    def __call__(self, *args, **kwargs):
        return self.func(*args, **kwargs)

@dataclass
class ContenedorEmbedding:
    func: callable
    max_token_size: int = 512
    embedding_dim: int = 1024

    def __call__(self, *args, **kwargs):
        return self.func(*args, **kwargs)

# 1. Definir la ruta de trabajo de forma segura
WORKING_DIR = "/content/drive/MyDrive/Colab Notebooks/knowledge_graph2"

# 2. Configurar tu cliente de Cohere con tu API Key
COHERE_API_KEY = "cohere_vjtwdYUlAkffQ4rvqDUO2pTGEzLVkfZrZyaXf6Zn4LaBJj"
co = cohere.Client(COHERE_API_KEY)

# 3. Función LLM nativa
async def cohere_llm_core(prompt, system_prompt=None, history_messages=[], **kwargs) -> str:
    chat_history = []
    current_message = prompt

    if system_prompt:
        # Prepend system prompt to the user's message as Cohere's chat API handles system messages
        # differently or might not have a dedicated 'system' role in chat_history for all models.
        current_message = f"{system_prompt}\n\n{prompt}"

    for msg in history_messages:
        role = msg.get("role", "").upper()
        # Cohere chat history roles are typically 'USER' and 'CHATBOT'
        # Convert 'user' to 'USER' and 'assistant' to 'CHATBOT'
        if role == "USER":
            chat_history.append({"role": "USER", "message": msg.get("content", "")})
        elif role == "ASSISTANT": # Assuming 'assistant' role for chatbot responses
            chat_history.append({"role": "CHATBOT", "message": msg.get("content", "")})
        # Other roles (like 'system') are handled by prepending to current_message or ignored for chat_history

    loop = asyncio.get_event_loop()
    retries = 5
    for attempt in range(retries):
        try:
            await asyncio.sleep(0.2) # Small delay before attempting
            response = await loop.run_in_executor(
                None,
                lambda: co.chat(
                    model="command-r-plus-08-2024",  # Updated model name to a supported version
                    message=current_message, # Pass the current prompt as 'message'
                    chat_history=chat_history, # Pass the history as 'chat_history'
                    temperature=kwargs.get("temperature", 0.7)
                )
            )
            return response.text # Cohere's chat response usually has the generated text in .text
        except Exception as e:
            if "429" in str(e) and attempt < retries - 1:
                wait_time = (attempt + 1) * 2
                print(f"⚠️ API ocupada. Reintentando en {wait_time}s...")
                await asyncio.sleep(wait_time)
            else:
                raise e

# 4. Función de embeddings nativa
async def cohere_embedding_core(texts: list[str]) -> list[list[float]]:
    loop = asyncio.get_event_loop()
    response = await loop.run_in_executor(
        None,
        lambda: co.embed(
            texts=texts,
            model="embed-multilingual-v3.0",
            input_type="search_document"
        )
    )
    return response.embeddings

# Envolvemos tus funciones en estos contenedores hechos a medida
llm_wrapped = ContenedorLLM(func=cohere_llm_core)
embedding_wrapped = ContenedorEmbedding(func=cohere_embedding_core)

# 6. Inicializar el pipeline pasando los contenedores que pide la versión 1.4.9.11
if not os.path.exists(WORKING_DIR):
    os.makedirs(WORKING_DIR, exist_ok=True)

rag = LightRAG(
    working_dir=WORKING_DIR,
    llm_model_func=llm_wrapped,
    embedding_func=embedding_wrapped
)

# Initialize storages as required by LightRAG
await rag.initialize_storages()

print("¡LightRAG-HKU inicializado con éxito usando contenedores dinámicos compatibles!")

INFO: [] Created new empty graph file: /content/drive/MyDrive/Colab Notebooks/knowledge_graph2/graph_chunk_entity_relation.graphml
INFO: [] Process 5225 KV load full_docs with 0 records
INFO: [] Process 5225 KV load text_chunks with 0 records
INFO: [] Process 5225 KV load full_entities with 0 records
INFO: [] Process 5225 KV load full_relations with 0 records
INFO: [] Process 5225 KV load entity_chunks with 0 records
INFO: [] Process 5225 KV load relation_chunks with 0 records
INFO: [] Process 5225 KV load llm_response_cache with 0 records
INFO: [] Process 5225 doc status load doc_status with 0 records


¡LightRAG-HKU inicializado con éxito usando contenedores dinámicos compatibles!


In [ ]:
import json
import asyncio
import os
from asyncio import CancelledError

def buscar_directorio_datos():
    # Intentar encontrar la carpeta que contiene los JSONs de LightRAG
    posibles_rutas = [
        "/content/drive/MyDrive/Colab Notebooks/knowledge_graph2",
        "/content/drive/MyDrive/Colab Notebooks/knowledge_graph",
        "/content/drive/MyDrive/knowledge_graph"
    ]

    for ruta in posibles_rutas:
        if os.path.exists(os.path.join(ruta, "kv_store_text_chunks.json")):
            return ruta
    return None

async def forzar_extraccion_del_grafo():
    global WORKING_DIR

    directorio_real = buscar_directorio_datos()

    if directorio_real:
        WORKING_DIR = directorio_real
        print(f"✅ Carpeta de datos detectada en: {WORKING_DIR}")
    else:
        print("❌ No se encontró 'kv_store_text_chunks.json' en las rutas conocidas.")
        print("Por favor, asegúrate de que la carpeta del grafo esté en tu Drive.")
        return

    # 1. Cargar los fragmentos de texto
    path_chunks = os.path.join(WORKING_DIR, "kv_store_text_chunks.json")
    with open(path_chunks, "r", encoding="utf-8") as f:
        chunks_data = json.load(f)

    textos_a_procesar = []
    for key, value in chunks_data.items():
        if isinstance(value, dict) and "content" in value:
            textos_a_procesar.append(value["content"])
        elif isinstance(value, str):
            textos_a_procesar.append(value)

    total_chunks = len(textos_a_procesar)
    print(f"📥 Fragmentos encontrados: {total_chunks}")

    if total_chunks == 0:
        return

    print("\n🚀 Iniciando extracción con Cohere (esto tomará tiempo)...\n")

    max_retries = 3
    for attempt in range(max_retries):
        try:
            # Usamos el objeto 'rag' inicializado anteriormente
            await rag.ainsert(textos_a_procesar)
            print(f"\n✅ Procesamiento completado en el intento {attempt + 1}.")
            break
        except CancelledError as e:
            if attempt < max_retries - 1:
                wait_time = (attempt + 1) * 5
                print(f"⚠️ Tiempo de espera agotado. Reintentando en {wait_time}s...")
                await asyncio.sleep(wait_time)
            else:
                raise e
        except Exception as e:
            print(f"❌ Error durante la inserción: {e}")
            break

    print("\n✨ ¡Proceso finalizado! El archivo .graphml debería estar actualizado en tu Drive.")

# Ejecutar el proceso
await forzar_extraccion_del_grafo()

✅ Carpeta de datos detectada en: /content/drive/MyDrive/Colab Notebooks/knowledge_graph
📥 Fragmentos encontrados: 759

🚀 Iniciando extracción con Cohere (esto tomará tiempo)...



INFO: Processing 759 document(s)
INFO: Extracting stage 1/759: unknown_source
INFO: Processing d-id: doc-f551b70ffa3410d267e6b4bd743d908c
INFO: Extracting stage 2/759: unknown_source
INFO: Processing d-id: doc-7f841e5333a40ec1b310d8f2c97afcf1
INFO: Embedding func: 8 new workers initialized (Timeouts: Func: 30s, Worker: 60s, Health Check: 75s)
INFO: LLM func: 4 new workers initialized (Timeouts: Func: 180s, Worker: 360s, Health Check: 375s)
INFO:  == LLM cache == saving: default:extract:6a5392cfc6d8ae13845ffbce7f5f9b5f
INFO:  == LLM cache == saving: default:extract:a509a5b8b0d266d01ba07c5c5b56cf93
INFO: Chunk 1 of 1 extracted 6 Ent + 0 Rel chunk-f551b70ffa3410d267e6b4bd743d908c
INFO: Merging stage 1/759: unknown_source
INFO: Phase 1: Processing 6 entities from doc-f551b70ffa3410d267e6b4bd743d908c (async: 8)
INFO: Phase 2: Processing 0 relations from doc-f551b70ffa3410d267e6b4bd743d908c (async: 8)
INFO: Phase 3: Updating final 6(6+0) entities and  0 relations from doc-f551b70ffa3410d267e

⚠️ Tiempo de espera agotado. Reintentando en 5s...


INFO: Reset 6 documents from PROCESSING/FAILED to PENDING status
INFO: Processing 757 document(s)
INFO: Extracting stage 1/757: unknown_source
INFO: Processing d-id: doc-616e53950265280af69e39b688a87bc8
INFO: Extracting stage 2/757: unknown_source
INFO: Processing d-id: doc-aab05ff311473a92895d2f7e4cb5292d
INFO: Chunk 1 of 2 extracted 11 Ent + 0 Rel chunk-067078c8cd7f95ead3e9dffced3fb516
INFO:  == LLM cache == saving: default:extract:4f65a55389dfb2a33dd8c1cd9ec2e362
INFO:  == LLM cache == saving: default:extract:721eac4f834e0f703cdf66f0789c7b15
INFO:  == LLM cache == saving: default:extract:721eac4f834e0f703cdf66f0789c7b15
INFO:  == LLM cache == saving: default:extract:f121ed588df5250e242216883a8734d0
INFO: Chunk 2 of 2 extracted 57 Ent + 0 Rel chunk-fb554bbdad89e67833af1f94f7c0b23d
INFO:  == LLM cache == saving: default:extract:61e7ace5ce0f760dfbf4d50d3d29dcb0
INFO: Chunk 1 of 2 extracted 8 Ent + 5 Rel chunk-3ee7a21108e37588a85aa33b0af5247a
INFO:  == LLM cache == saving: default:extra